# 🧠 MS Lesion Segmentation: Training Notebook
### Stage 1: Infrastructure & Data Ingestion

This notebook is designed to download raw MS MRI datasets and prepare the environment for 3D Deep Learning.

## 🛠 1.1 Environment Setup
Install core medical AI libraries and mount Google Drive for saving model weights.

In [ ]:
!pip install -q monai[all] SimpleITK nibabel tqdm

from google.colab import drive
import os

drive.mount('/content/drive')

# Define Workspace Paths
BASE_DIR = "/content"
DATA_DIR = os.path.join(BASE_DIR, "data/raw")
PROCESSED_DIR = os.path.join(BASE_DIR, "data/processed")
MODEL_SAVE_PATH = "/content/drive/MyDrive/MS_Project/models"

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(MODEL_SAVE_PATH, exist_ok=True)

print(f"✅ Workspace initialized. Models will be saved to: {MODEL_SAVE_PATH}")

## 📊 1.2 Data Ingestion (Hybrid Strategy)
We pull the ZIP files from your **Google Drive** and extract them to **local Colab storage** (`/content/data`) for maximum training speed.

In [ ]:
import os
import glob
from google.colab import drive

# 1. Setup Paths
drive.mount('/content/drive')
DRIVE_DATA_ROOT = "/content/drive/MyDrive/brain_dataset"
LOCAL_DATA_DIR = "/content/data/raw"

os.makedirs(LOCAL_DATA_DIR, exist_ok=True)

# 2. Categories to Extract
categories = ["MSLesSeg", "Mendeley", "Long-MR-MS"]

for category in categories:
    drive_folder = os.path.join(DRIVE_DATA_ROOT, category)
    local_folder = os.path.join(LOCAL_DATA_DIR, category)
    os.makedirs(local_folder, exist_ok=True)
    
    print(f"\n📦 Processing {category}...")
    zips = glob.glob(os.path.join(drive_folder, "*.zip"))
    
    if not zips:
        print(f"  ⚠️ No ZIP files found in {drive_folder}")
        continue
        
    for zip_path in zips:
        print(f"  ⚡ Extracting {os.path.basename(zip_path)} to local disk...")
        !unzip -q -o "{zip_path}" -d "{local_folder}"

print("\n🚀 SUCCESS: Fast local data storage is ready for training.")

## 🔍 1.2.1 Directory Mapping (Diagnostic)
Run this cell to see exactly where your FLAIR and Mask files are located inside the subfolders.

In [ ]:
import os

def map_directory_structure(root_dir, max_depth=3):
    print(f"🔍 Mapping structure for: {root_dir}\n")
    for root, dirs, files in os.walk(root_dir):
        depth = root.replace(root_dir, '').count(os.sep)
        if depth <= max_depth:
            indent = '  ' * depth
            print(f"{indent}📂 {os.path.basename(root)}/")
            if files:
                for f in files[:3]:
                    print(f"{indent}  📄 {f}")
                if len(files) > 3:
                    print(f"{indent}  ... ({len(files)-3} more files)")

map_directory_structure("/content/data/raw")

## 📂 1.2.2 Data Discovery & Pairing
This cell pairs MRI scans with their Ground Truth masks for training across all datasets.

In [ ]:
import glob

def create_data_list(raw_dir):
    data_list = []
    
    # 1. Long-MR-MS (Patient-based matching)
    long_path = os.path.join(raw_dir, 'Long-MR-MS')
    if os.path.exists(long_path):
        patient_folders = glob.glob(os.path.join(long_path, 'patient*'))
        for p_folder in patient_folders:
            gt = glob.glob(os.path.join(p_folder, '*_gt.nii.gz'))
            flairs = glob.glob(os.path.join(p_folder, '*_FLAIRreg.nii.gz'))
            if gt and flairs:
                for f in flairs:
                    data_list.append({'image': f, 'label': gt[0]})
    
    # 2. Handle Mendeley (Strict exclusion of masks as images)
    mendeley_path = os.path.join(raw_dir, 'Mendeley')
    if os.path.exists(mendeley_path):
        flair_imgs = [f for f in glob.glob(os.path.join(mendeley_path, '**/*-Flair.nii'), recursive=True) 
                      if "LESIONSEG" not in f.upper()]
        for f in flair_imgs:
            p_id = os.path.basename(f).split('-')[0]
            mask = os.path.join(os.path.dirname(f), f"{p_id}-LesionSeg-Flair.nii")
            if os.path.exists(mask):
                data_list.append({'image': f, 'label': mask})
    
    # 3. Handle MSLesSeg (Timepoint-Aware Matching)
    msles_path = os.path.join(raw_dir, 'MSLesSeg')
    if os.path.exists(msles_path):
        all_nii = glob.glob(os.path.join(msles_path, '**/*.nii*'), recursive=True)
        mask_map = {}
        for m in all_nii:
            if "_MASK" in m.upper():
                parts = os.path.basename(m).split('_')
                if len(parts) >= 2: mask_map[f"{parts[0]}_{parts[1]}"] = m
        
        flairs = [f for f in all_nii if "FLAIR" in f.upper() and "_MASK" not in f.upper()]
        for f in flairs:
            parts = os.path.basename(f).split('_')
            if len(parts) >= 2:
                key = f"{parts[0]}_{parts[1]}"
                if key in mask_map: data_list.append({'image': f, 'label': mask_map[key]})
    
    print(f"✅ Total Honest Pairs Discovered: {len(data_list)}")
    return data_list

train_files = create_data_list("/content/data/raw")

## 🧠 1.3 The Preprocessing Engine
Standardizing MRI volumes to 1mm isotropic spacing and RAS orientation.

In [ ]:
import SimpleITK as sitk
import numpy as np

def preprocess_volume(img_path, target_spacing=(1.0, 1.0, 1.0), is_mask=False):
    img = sitk.ReadImage(img_path)
    img = sitk.DICOMOrient(img, "RAS")
    
    # Standardize Voxel Spacing
    original_spacing = img.GetSpacing()
    original_size = img.GetSize()
    new_size = [int(round(osz * ospc / tspc)) for osz, ospc, tspc in zip(original_size, original_spacing, target_spacing)]
    
    resampler = sitk.ResampleImageFilter()
    resampler.SetOutputSpacing(target_spacing)
    resampler.SetSize(new_size)
    resampler.SetOutputOrigin(img.GetOrigin())
    resampler.SetOutputDirection(img.GetDirection())
    resampler.SetInterpolator(sitk.sitkNearestNeighbor if is_mask else sitk.sitkBSpline)
    img = resampler.Execute(img)
    
    if not is_mask:
        # Fix lighting (N4)
        mask = sitk.OtsuThreshold(img, 0, 1)
        corrector = sitk.N4BiasFieldCorrectionImageFilter()
        img = corrector.Execute(img, mask)
        # Normalize intensities
        array = sitk.GetArrayFromImage(img)
        array = (array - np.mean(array)) / (np.std(array) + 1e-8)
        img_out = sitk.GetImageFromArray(array)
        img_out.CopyInformation(img)
        return img_out
    return img

print("✅ Preprocessing logic ready.")

## 🧹 1.3.1 Bulk Preprocessing (MANDATORY)
This cell standardizes all 100+ raw patients and saves them to `/content/data/processed`. This ensures 100% stability and higher accuracy.

In [ ]:
processed_train_files = []
print(f"🚀 Cleaning and Standardizing {len(train_files)} pairs. Please wait...")

for i, pair in enumerate(tqdm(train_files, desc="Standardizing Data")):
    subj_id = f"sub-{i:03d}"
    subj_dir = os.path.join(PROCESSED_DIR, subj_id)
    os.makedirs(subj_dir, exist_ok=True)
    
    img_out_path = os.path.join(subj_dir, "image.nii.gz")
    lbl_out_path = os.path.join(subj_dir, "label.nii.gz")
    
    # Preprocess if not already done
    if not os.path.exists(img_out_path):
        img_clean = preprocess_volume(pair['image'], is_mask=False)
        lbl_clean = preprocess_volume(pair['label'], is_mask=True)
        sitk.WriteImage(img_clean, img_out_path)
        sitk.WriteImage(lbl_clean, lbl_out_path)
    
    processed_train_files.append({'image': img_out_path, 'label': lbl_out_path})

print(f"\n✅ ALL DATA CLEANED: Standardized files stored in {PROCESSED_DIR}")

## 🧪 1.4 Architecture: 3D Residual U-Net
We implement a 3D Residual U-Net using MONAI, optimized for small batch sizes with InstanceNorm.

In [ ]:
from monai.networks.nets import UNet
import torch

def get_model(in_channels=1, out_channels=1):
    model = UNet(
        spatial_dims=3,
        in_channels=in_channels,
        out_channels=out_channels,
        channels=(16, 32, 64, 128, 256),
        strides=(2, 2, 2, 2),
        num_res_units=2,
        norm="INSTANCE",
        act="LEAKYRELU",
        dropout=0.1
    )
    return model

print("✅ 3D U-Net Architecture loaded.")

## 📉 1.5 Loss & Metrics
Hybrid Dice + Weighted BCE for small-lesion sensitivity.

In [ ]:
from monai.losses import DiceLoss
import torch.nn as nn
from scipy.ndimage import label

class MSLesionLoss(nn.Module):
    def __init__(self, alpha=1.0, weight=10.0):
        super(MSLesionLoss, self).__init__()
        self.alpha = alpha
        self.dice_loss = DiceLoss(sigmoid=True, squared_pred=True)
        self.bce_loss = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([weight]).cuda())

    def forward(self, input, target):
        return self.dice_loss(input, target) + self.alpha * self.bce_loss(input, target)

print("✅ Loss and Metrics logic implemented.")

## 🚀 1.6 The Full Training Engine
Executes the actual 3D training loop with Balanced Sampling and AMP.

In [ ]:
from monai.data import PersistentDataset, DataLoader, pad_list_data_collate
from monai.transforms import (Compose, LoadImaged, EnsureChannelFirstd, SpatialPadd, RandCropByPosNegLabeld, ToTensord)
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm.notebook import tqdm
import time

def train_model(data_list, epochs=50):
    device = torch.device("cuda")
    model = get_model().to(device)
    loss_fn = MSLesionLoss().to(device)
    optimizer = AdamW(model.parameters(), lr=1e-4)
    scheduler = CosineAnnealingLR(optimizer, T_max=epochs)
    scaler = torch.amp.GradScaler('cuda')
    
    # Now using simplified transforms because data is already pre-cleaned!
    transforms = Compose([
        LoadImaged(keys=["image", "label"]),
        EnsureChannelFirstd(keys=["image", "label"]),
        SpatialPadd(keys=["image", "label"], spatial_size=(96, 96, 96)),
        RandCropByPosNegLabeld(keys=["image", "label"], label_key="label", spatial_size=(96,96,96), pos=1, neg=1, num_samples=4),
        ToTensord(keys=["image", "label"])
    ])
    
    persist_dir = "/content/persistent_cache"
    if os.path.exists(persist_dir): shutil.rmtree(persist_dir)
    os.makedirs(persist_dir, exist_ok=True)
    
    ds = PersistentDataset(data=data_list, transform=transforms, cache_dir=persist_dir)
    loader = DataLoader(ds, batch_size=1, shuffle=True, collate_fn=pad_list_data_collate)
    
    print("🚀 Starting Robust Training Engine on T4 GPU...")
    start_time_total = time.time()
    
    for epoch in range(epochs):
        start_time_epoch = time.time()
        model.train()
        epoch_loss = 0
        total_correct, total_pixels = 0, 0
        
        for batch in tqdm(loader, desc=f"Epoch {epoch+1}/{epochs}"):
            images, labels = batch["image"].to(device), batch["label"].to(device)
            optimizer.zero_grad()
            with torch.amp.autocast('cuda'):
                outputs = model(images)
                loss = loss_fn(outputs, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            epoch_loss += loss.item()
            
            # Accuracy tracking
            preds = (outputs.sigmoid() > 0.5).float()
            total_correct += (preds == labels).sum().item()
            total_pixels += labels.numel()
        
        scheduler.step()
        accuracy = total_correct / total_pixels
        epoch_duration = time.time() - start_time_epoch
        print(f"🔥 Epoch {epoch+1} | Loss: {epoch_loss/len(loader):.4f} | Acc: {accuracy:.4%} | Time: {epoch_duration/60:.2f}m")
        
        if (epoch + 1) % 5 == 0:
            torch.save(model.state_dict(), os.path.join(MODEL_SAVE_PATH, "best_model.pth"))
            print(f"💾 Checkpoint saved to Google Drive.")

    total_duration = time.time() - start_time_total
    print(f"\n✅ Training Complete in {total_duration/60:.2f} mins.")

print("✅ Optimized Training Engine Ready.")

# train_model(processed_train_files)